# RAC Pipeline — Inference Latency

Measures end-to-end latency and throughput of the full RAC pipeline (SBERT retrieval → RoBERTa classifier) in samples/sec and GPU memory.

## 1. Imports

In [ ]:
import sys
import time
import json
import torch
import faiss
from pathlib import Path
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

sys.path.insert(0, str(Path("..").resolve()))
from retriever import retrieve_top_k_above_threshold
from data_loaders import load_ihc_binary

## 2. Configuration

Edit the variables below to change which model and index are benchmarked,
or to adjust the number of warmup and timed samples.

In [ ]:
# Define paths
ROOT_DIR        = Path("../..")
INDEX_DIR       = ROOT_DIR / "corpus" / "index"
CLASSIFIER_DIR  = ROOT_DIR / "weigths" / "weights_rac_best_hyperparameters" / "roberta" / "sbert" / "full" / "IHC"
RETRIEVER_HF_ID = "sentence-transformers/all-mpnet-base-v2"

# Retrieval and classifier config
K          = 5
THRESHOLD  = 0.4
MAX_LENGTH = 256
# Benchmark config
N_WARMUP   = 50    # samples discarded before timing
N_SAMPLES  = 500   # samples timed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 3. Load Models and Index

Load the SBERT retriever, the FAISS full index, and the RoBERTa classifier.

In [ ]:
# Load SBERT retriever
print(f"Loading retriever: {RETRIEVER_HF_ID} ...")
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)

# Load FAISS full index
print("Loading FAISS full index ...")
ret_index = faiss.read_index(str(INDEX_DIR / "vdb_full.faiss"))
with open(INDEX_DIR / "lookup_full.json") as f:
    ret_documents = json.load(f)
print(f"  {ret_index.ntotal:,} vectors")

# Load RAC classifier
print(f"Loading RAC classifier from {CLASSIFIER_DIR} ...")
clf_tokenizer = AutoTokenizer.from_pretrained(str(CLASSIFIER_DIR))
clf_model     = AutoModelForSequenceClassification.from_pretrained(
    str(CLASSIFIER_DIR), num_labels=2
).eval().to(device)
print("Models ready.")

## 4. Load Test Data

Load the IHC test split using `load_ihc_binary`. Only the first
`N_WARMUP + N_SAMPLES` texts are used.

In [ ]:
_, test_ihc = load_ihc_binary(seed=42)
test_texts  = [ex["post"] for ex in test_ihc]
samples     = test_texts[:N_WARMUP + N_SAMPLES]
print(f"Using {len(samples)} samples ({N_WARMUP} warmup + {N_SAMPLES} timed)")

## 5. Single-Sample Inference

The `infer_single` function runs the full pipeline for one text:
retrieve neighbors → build augmented string → classify.

In [ ]:
def infer_single(text):
    # retrieve neighbors then build augmented input and classify
    neighbors = retrieve_top_k_above_threshold(
        text, THRESHOLD, ret_model, ret_tokenizer,
        ret_index, ret_documents,
        chunk_id=None, k=K, use_mean_pool=True,
    )
    neighbor_texts = [t for t, _ in neighbors]

    sep       = clf_tokenizer.sep_token
    augmented = f" {sep} ".join([text] + neighbor_texts)
    inputs    = clf_tokenizer(
        augmented,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        logits = clf_model(**inputs).logits
    return int(torch.argmax(logits, dim=-1).item())

## 6. Warmup and Timed Run

The first `N_WARMUP` samples are discarded to warm up the GPU cache.
The following `N_SAMPLES` samples are then timed.

In [ ]:
# warmup to fill GPU cache
print(f"Warming up ({N_WARMUP} samples) ...")
for text in samples[:N_WARMUP]:
    infer_single(text)
if device.type == "cuda":
    torch.cuda.synchronize()

if device.type == "cuda":
    mem_gb          = torch.cuda.memory_allocated(device) / 1024**3
    mem_reserved_gb = torch.cuda.memory_reserved(device) / 1024**3
    print(f"GPU memory allocated : {mem_gb:.2f} GB")
    print(f"GPU memory reserved  : {mem_reserved_gb:.2f} GB")
else:
    mem_gb = mem_reserved_gb = None
    print("(CPU only — no GPU memory to report)")

# timed run
print(f"\nTiming {N_SAMPLES} samples ...")
timed_samples = samples[N_WARMUP:]

if device.type == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()

for text in timed_samples:
    infer_single(text)

if device.type == "cuda":
    torch.cuda.synchronize()
t1 = time.perf_counter()

elapsed    = t1 - t0
throughput = N_SAMPLES / elapsed

## 7. Results

In [ ]:
# print throughput results
print(f"Elapsed          : {elapsed:.2f} s")
print(f"Throughput       : {throughput:.1f} samples/sec")
print(f"Latency/sample   : {1000 * elapsed / N_SAMPLES:.1f} ms")
if mem_gb is not None:
    print(f"GPU memory       : {mem_gb:.2f} GB allocated / {mem_reserved_gb:.2f} GB reserved")

# Twitter-scale back-of-envelope estimate
tweets_per_day = 500_000_000
tweets_per_sec = tweets_per_day / 86_400
gpus_needed    = tweets_per_sec / throughput

print(f"\nTwitter-scale estimate ({tweets_per_day/1e6:.0f}M tweets/day):")
print(f"  Tweets/sec : {tweets_per_sec:.0f}")
print(f"  GPUs needed: {gpus_needed:.1f}")